# Inheriting from `type`
## Advanced Problems with Solutions — Tutorial Style, Set 2

This notebook is a **new set of advanced exercises** on the same topic: creating custom metaclasses by inheriting from `type`.

The style intentionally mirrors a lecture/tutorial notebook:

- we introduce one idea at a time,
- we break each larger problem into small logical steps,
- we frequently inspect intermediate objects,
- we pause to predict behavior before running code,
- and only then do we assemble the final solution.

The goal is not merely to produce working metaclasses.

The goal is to understand **why each hook exists, what object it receives, and what part of Python's class-creation machinery it controls**.

### What we will build

We will solve a sequence of increasingly advanced problems:

1. inject derived class metadata,
2. validate class definitions before they can exist,
3. intercept class-body assignments with `__prepare__`,
4. control instance creation with metaclass `__call__`,
5. build an inheritance-aware command registry,
6. generate descriptors from declarative field definitions,
7. wrap methods during class creation,
8. compose custom metaclasses with `ABCMeta`,
9. consume custom class-definition keywords,
10. build a small record framework,
11. compare metaclasses with simpler alternatives.

Every major problem contains a complete solution and executable checks.

### Imports

We will keep dependencies inside the Python standard library so the notebook remains self-contained.

In [1]:
from abc import ABC, ABCMeta, abstractmethod
from collections import OrderedDict
from types import MappingProxyType
import functools
import inspect

# 1. Refresher: a class is an object too

Before solving advanced problems, we need the right mental model.

When Python executes:

```python
class Example:
    x = 10
```

it eventually needs to construct an object representing the class `Example`.

By default, the object responsible for constructing that class object is `type`.

In [2]:
class Example:
    x = 10

type(Example)

type

The result is `type`.

That means `Example` is an **instance of `type`**.

This is exactly parallel to ordinary objects:

```python
obj = Example()
```

Here `obj` is an instance of `Example`.

But one level higher:

```python
Example
```

is itself an object, and its class is `type`.

In [3]:
obj = Example()

(type(obj), type(Example))

(__main__.Example, type)

So if we create our own subclass of `type`, we can customize the creation of class objects.

That subclass is what we usually call a **metaclass**.

In [4]:
class DemoMeta(type):
    pass

class Demo(metaclass=DemoMeta):
    pass

type(Demo)

__main__.DemoMeta

The class `Demo` behaves like a normal class, but its creator is now `DemoMeta`.

That gives us a place to intercept class creation.

# 2. Problem 1 — injecting derived class metadata

Suppose we are building a framework in which every class should expose a human-readable label.

We want this:

```python
class HTTPClient(metaclass=LabelMeta):
    pass
```

to automatically receive something like:

```python
HTTPClient.label == "HTTP Client"
```

The class author should not need to write that label manually.

### Step 1 — what information does metaclass `__new__` receive?

A typical metaclass `__new__` looks like this:

```python
def __new__(mcls, name, bases, namespace, **kwargs):
    ...
```

The important values are:

- `mcls`: the metaclass currently creating the class,
- `name`: the name of the class being created,
- `bases`: its base classes,
- `namespace`: the dictionary produced by executing its class body.

For this problem, the class name itself is enough information to derive the label.

So we can modify the namespace **before** delegating to `type`.

### Step 2 — write a helper first

Metaclass code is easier to test when transformations are separated into normal functions.

We will convert CamelCase names into spaced words.

In [5]:
def camel_to_words(name):
    result = []

    for index, char in enumerate(name):
        if (
            index > 0
            and char.isupper()
            and not name[index - 1].isupper()
        ):
            result.append(" ")

        result.append(char)

    return "".join(result)

In [6]:
camel_to_words("HTTPClient"), camel_to_words("UserAccount")

('HTTPClient', 'User Account')

`HTTPClient` is slightly tricky because the acronym is already uppercase.

Our simple helper produces a reasonable teaching example, but a production-quality name splitter might use a more sophisticated regular expression.

For metaclass design, the important point is that the transformation is kept **outside** the metaclass.

### Step 3 — inject the label

We should not overwrite a label explicitly supplied by the class author.

So we use `setdefault`.

In [7]:
class LabelMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        namespace.setdefault(
            "label",
            camel_to_words(name),
        )

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

Now we can try it on two classes:

- one that relies on the generated value,
- one that explicitly provides its own value.

In [8]:
class UserAccount(metaclass=LabelMeta):
    pass


class APIClient(metaclass=LabelMeta):
    label = "External API Client"


UserAccount.label, APIClient.label

('User Account', 'External API Client')

### What should we notice?

The metaclass is modifying the **class definition**, not the instances.

No instance needs to be created for the new attribute to exist.

In [9]:
"label" in UserAccount.__dict__, UserAccount().label

(True, 'User Account')

### Solution pattern

This is one of the simplest useful metaclass patterns:

1. inspect the namespace,
2. add or transform entries,
3. delegate to `super().__new__`,
4. return the resulting class object.

# 3. Problem 2 — rejecting invalid class definitions

Now we want a stricter framework.

Every concrete event class must define:

```python
event_type = "some_name"
```

We want an error immediately when an invalid class is defined.

That is earlier and clearer than waiting until somebody creates an instance.

### Step 1 — define the contract

Our rules will be:

- the framework base class is exempt,
- every concrete subclass must define `event_type`,
- the value must be a non-empty string,
- it must contain only lowercase letters, digits, and underscores.

We will first write a helper for the naming rule.

In [10]:
def is_event_name(value):
    if not isinstance(value, str) or not value:
        return False

    return all(
        char.islower() or char.isdigit() or char == "_"
        for char in value
    )

In [11]:
[
    (value, is_event_name(value))
    for value in ["user_created", "OrderCreated", "", 123]
]

[('user_created', True), ('OrderCreated', False), ('', False), (123, False)]

### Step 2 — validate inside metaclass `__new__`

The class has not been created yet, so this is an ideal place to reject invalid definitions.

In [12]:
class EventMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        is_framework_base = not bases

        if not is_framework_base:
            if "event_type" not in namespace:
                raise TypeError(
                    f"{name} must define event_type"
                )

            event_type = namespace["event_type"]

            if not is_event_name(event_type):
                raise ValueError(
                    f"Invalid event_type: {event_type!r}"
                )

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

### Step 3 — define the base and a valid subclass

In [13]:
class Event(metaclass=EventMeta):
    pass


class UserCreated(Event):
    event_type = "user_created"


UserCreated.event_type

'user_created'

### Step 4 — inspect a failure

A useful framework should fail with a clear message.

We will intentionally define a bad class inside `exec` so the notebook can continue after the exception.

In [14]:
try:
    exec(
        '''
class BrokenEvent(Event):
    pass
''',
        globals(),
        {},
    )
except TypeError as exc:
    event_error = str(exc)

event_error

'BrokenEvent must define event_type'

The error appears at **class-definition time**.

This is an important metaprogramming advantage: we can prevent malformed classes from entering the program at all.

### Extension — protect uniqueness too

A natural next requirement is to ensure no two concrete event classes use the same `event_type`.

That requires state shared across class creations.

The metaclass itself is a convenient place to keep that registry.

In [15]:
class UniqueEventMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        if bases:
            event_type = namespace.get("event_type")

            if not is_event_name(event_type):
                raise ValueError(
                    f"{name} needs a valid event_type"
                )

            if event_type in mcls.registry:
                raise ValueError(
                    f"Duplicate event_type: {event_type!r}"
                )

            mcls.registry[event_type] = cls

        return cls

Notice the slight design difference.

Here we create the class first and register it afterward.

That is useful when registration wants access to the finished class object.

# 4. Problem 3 — detecting duplicate names with `__prepare__`

Consider this class:

```python
class Settings:
    timeout = 10
    timeout = 20
```

Python normally accepts it.

By the time class creation finishes, only the second value remains.

In [16]:
class NormalSettings:
    timeout = 10
    timeout = 20

NormalSettings.timeout

20

Suppose we are designing a configuration language where duplicate definitions should be illegal.

Can metaclass `__new__` detect the duplicate?

With a normal dictionary, no.

By the time `__new__` sees the namespace, the first value has already been overwritten.

### Step 1 — intercept assignments earlier

This is what `__prepare__` is for.

`__prepare__` returns the mapping object Python will use while executing the class body.

So we need a custom dictionary that rejects reassignment.

In [17]:
class DuplicateRejectingDict(dict):
    def __setitem__(self, key, value):
        if key in self and not key.startswith("__"):
            raise TypeError(
                f"Duplicate class attribute: {key!r}"
            )

        super().__setitem__(key, value)

### Step 2 — make the metaclass return that mapping

In [18]:
class StrictNamespaceMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return DuplicateRejectingDict()

We have not even overridden `__new__`.

That is perfectly valid.

The normal `type.__new__` can consume the resulting mapping after the class body finishes.

### Step 3 — test a valid class

In [19]:
class StrictSettings(metaclass=StrictNamespaceMeta):
    timeout = 30
    retries = 3


StrictSettings.timeout, StrictSettings.retries

(30, 3)

### Step 4 — test a duplicate

The error is raised **while the class body is executing**, before metaclass `__new__` would even run.

In [20]:
try:
    exec(
        '''
class InvalidSettings(metaclass=StrictNamespaceMeta):
    timeout = 10
    timeout = 20
''',
        globals(),
        {},
    )
except TypeError as exc:
    duplicate_name_error = str(exc)

duplicate_name_error

"Duplicate class attribute: 'timeout'"

### Important distinction

Use:

- `__prepare__` when you need to control or observe the namespace **during class-body execution**,
- `__new__` when you need to transform or validate the completed namespace **after class-body execution**.

# 5. Problem 4 — controlling instance creation with metaclass `__call__`

So far we have customized how classes are created.

Now we will customize what happens when those classes are called.

Remember:

```python
SomeClass(...)
```

is effectively a call to the metaclass's `__call__`.

### Goal

We will build a flyweight-style cache.

For a class using our metaclass:

```python
Symbol("USD")
Symbol("USD")
```

should return the same object.

But:

```python
Symbol("EUR")
```

should return a different object.

### Step 1 — identify the correct hook

Metaclass `__new__` creates the **class object**.

Metaclass `__call__` controls calling that class object to produce instances.

Therefore instance reuse belongs in metaclass `__call__`.

In [21]:
class FlyweightMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.__instance_cache__ = {}
        return cls

    def __call__(cls, key, *args, **kwargs):
        if key not in cls.__instance_cache__:
            cls.__instance_cache__[key] = super().__call__(
                key,
                *args,
                **kwargs,
            )

        return cls.__instance_cache__[key]

### Step 2 — define a normal-looking class

The class itself does not need to know anything about caching.

In [22]:
class Symbol(metaclass=FlyweightMeta):
    def __init__(self, code):
        self.code = code

    def __repr__(self):
        return f"Symbol({self.code!r})"

### Step 3 — verify identity, not merely equality

In [23]:
usd_1 = Symbol("USD")
usd_2 = Symbol("USD")
eur = Symbol("EUR")

(usd_1 is usd_2, usd_1 is eur)

(True, False)

The first pair is literally the same object.

This shows how strongly a metaclass can influence instance construction.

### Step 4 — confirm the cache belongs to each class

Because `__instance_cache__` is attached to each newly created class, subclasses receive their own cache.

In [24]:
class CurrencySymbol(Symbol):
    pass


base_usd = Symbol("USD")
child_usd = CurrencySymbol("USD")

base_usd is child_usd

False

They are different because `Symbol` and `CurrencySymbol` maintain separate caches.

That is often the desired behavior.

If a framework wanted one cache shared across many classes, the cache could instead live directly on the metaclass.

# 6. Problem 5 — an inheritance-aware command registry

Registries are one of the most common practical uses of metaclasses.

We will build a command system.

A command class will look like:

```python
class SaveCommand(Command):
    command_name = "save"

    def execute(self, payload):
        ...
```

The framework should automatically remember every concrete command class.

### Requirements

Our registry should:

- ignore the abstract framework base,
- require a valid `command_name`,
- reject duplicate command names,
- expose a read-only view,
- allow lookup by name,
- work through inheritance.

### Step 1 — where should the registry live?

If the registry belongs to one class, inheritance may make ownership confusing.

We will store it directly on the metaclass.

That makes it clearly global to the metaclass-controlled framework.

In [25]:
class CommandMeta(type):
    _registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        if namespace.get("_abstract", False):
            return cls

        if bases:
            command_name = namespace.get("command_name")

            if not isinstance(command_name, str):
                raise TypeError(
                    f"{name} must define command_name"
                )

            if command_name in mcls._registry:
                raise ValueError(
                    f"Duplicate command: {command_name!r}"
                )

            mcls._registry[command_name] = cls

        return cls

    @classmethod
    def get(mcls, name):
        return mcls._registry[name]

    @classmethod
    def registry(mcls):
        return MappingProxyType(mcls._registry)

### Step 2 — define the framework base

We mark it abstract using a simple class-level flag.

Later we will solve a more formal abstract-base-class version.

In [26]:
class Command(metaclass=CommandMeta):
    _abstract = True

### Step 3 — add concrete commands

In [27]:
class SaveCommand(Command):
    _abstract = False
    command_name = "save"

    def execute(self, payload):
        return f"saved:{payload}"


class DeleteCommand(Command):
    _abstract = False
    command_name = "delete"

    def execute(self, payload):
        return f"deleted:{payload}"

The classes register themselves during their own creation.

In [28]:
dict(CommandMeta.registry())

{'save': __main__.SaveCommand, 'delete': __main__.DeleteCommand}

### Step 4 — instantiate by registry lookup

A registry becomes especially useful when input is data-driven.

In [29]:
command_cls = CommandMeta.get("save")
command = command_cls()

command.execute("document-17")

'saved:document-17'

### Step 5 — protect the registry from accidental mutation

`registry()` returns a `MappingProxyType`.

The caller can inspect the mapping but cannot assign new entries through that view.

In [30]:
registry_view = CommandMeta.registry()

try:
    registry_view["hack"] = object
except TypeError as exc:
    registry_readonly_error = type(exc).__name__

registry_readonly_error

'TypeError'

# 7. Problem 6 — generating descriptors from declarative fields

Now we will combine metaclasses with descriptors.

We want users of our framework to write:

```python
class Product(metaclass=SchemaMeta):
    fields = {
        "name": str,
        "price": float,
    }
```

and automatically receive validated attributes.

### Step 1 — build the descriptor independently

The descriptor should:

- remember the expected type,
- learn its public attribute name,
- store the real value under a private name,
- validate assignments.

In [31]:
class TypedField:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        return getattr(instance, self.storage_name)

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be "
                f"{self.expected_type.__name__}"
            )

        setattr(instance, self.storage_name, value)

The descriptor itself contains no metaclass logic.

That separation is useful.

The metaclass will only automate descriptor creation.

### Step 2 — transform the schema into descriptors

For every entry:

```python
"name": str
```

we inject:

```python
name = TypedField(str)
```

into the class namespace.

In [32]:
class SchemaMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        fields = namespace.get("fields", {})

        if not isinstance(fields, dict):
            raise TypeError("fields must be a dict")

        for field_name, field_type in fields.items():
            if not isinstance(field_type, type):
                raise TypeError(
                    f"{field_name!r} must map to a type"
                )

            namespace[field_name] = TypedField(field_type)

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

### Step 3 — define a class declaratively

In [33]:
class Product(metaclass=SchemaMeta):
    fields = {
        "name": str,
        "price": float,
    }

    def __init__(self, name, price):
        self.name = name
        self.price = price

The resulting attributes are descriptors.

In [34]:
type(Product.__dict__["name"]), type(Product.__dict__["price"])

(__main__.TypedField, __main__.TypedField)

### Step 4 — valid assignment

In [35]:
product = Product("Keyboard", 99.0)

product.name, product.price

('Keyboard', 99.0)

### Step 5 — invalid assignment

In [36]:
try:
    product.price = "expensive"
except TypeError as exc:
    descriptor_error = str(exc)

descriptor_error

'price must be float'

### Why this is a useful metaclass pattern

The user writes a compact declarative schema.

The metaclass translates that schema into richer Python behavior.

This pattern appears in ORMs, serializers, validation libraries, and API frameworks.

# 8. Problem 7 — wrapping public methods during class creation

Suppose a framework wants to count how often public instance methods are called.

We could decorate every method manually.

But if the rule applies to every public method of every framework class, a metaclass can centralize it.

### Step 1 — define exactly what should be wrapped

We will wrap only:

- ordinary functions defined directly in the class body,
- whose names do not begin with `_`.

We will skip:

- dunder methods,
- private helpers,
- `staticmethod`,
- `classmethod`,
- non-callable attributes.

### Step 2 — write the wrapper factory

A wrapper factory avoids the classic loop-closure bug.

In [37]:
def build_counting_wrapper(method_name, func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        counts = type(self).__method_call_counts__
        counts[method_name] = counts.get(method_name, 0) + 1

        return func(self, *args, **kwargs)

    return wrapper

`functools.wraps` preserves metadata such as the original function name and documentation.

### Step 3 — transform methods in the namespace

In [38]:
class CountingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace["__method_call_counts__"] = {}

        for attr_name, attr_value in list(namespace.items()):
            if attr_name.startswith("_"):
                continue

            if isinstance(
                attr_value,
                (staticmethod, classmethod),
            ):
                continue

            if inspect.isfunction(attr_value):
                namespace[attr_name] = build_counting_wrapper(
                    attr_name,
                    attr_value,
                )

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

### Step 4 — define a class normally

In [39]:
class Calculator(metaclass=CountingMeta):
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

    def _internal(self):
        return "hidden"

### Step 5 — call methods and inspect the class-level counters

In [40]:
calc = Calculator()

calc.add(1, 2)
calc.add(10, 20)
calc.multiply(3, 4)

Calculator.__method_call_counts__

{'add': 2, 'multiply': 1}

The counting behavior was injected once, at class creation.

Instances remain ordinary instances.

### Caution

This technique is powerful but implicit.

If only two or three methods need wrapping, explicit decorators are usually clearer.

Metaclass-wide wrapping becomes more attractive when the framework contract genuinely applies to many classes and methods.

# 9. Problem 8 — combining custom behavior with abstract base classes

Now we introduce a realistic complication.

Suppose our framework needs abstract methods:

```python
class Serializer(ABC):
    @abstractmethod
    def dumps(...):
        ...
```

But we also want automatic class registration.

`ABC` already relies on a metaclass: `ABCMeta`.

### Step 1 — understand the compatibility requirement

A custom metaclass used with `ABC` must be compatible with `ABCMeta`.

The easiest solution is usually to subclass `ABCMeta` instead of subclassing plain `type`.

In [41]:
class SerializerMeta(ABCMeta):
    registry = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        if bases and not cls.__abstractmethods__:
            mcls.registry[name] = cls

        return cls

### Step 2 — define the abstract framework

In [42]:
class Serializer(ABC, metaclass=SerializerMeta):
    @abstractmethod
    def dumps(self, obj):
        raise NotImplementedError

At this point `Serializer` is abstract, so it should not be registered.

In [43]:
dict(SerializerMeta.registry)

{}

### Step 3 — create concrete implementations

In [44]:
class TextSerializer(Serializer):
    def dumps(self, obj):
        return str(obj)


class ReprSerializer(Serializer):
    def dumps(self, obj):
        return repr(obj)

In [45]:
dict(SerializerMeta.registry)

{'TextSerializer': __main__.TextSerializer,
 'ReprSerializer': __main__.ReprSerializer}

Only concrete subclasses are registered.

The abstract-class machinery and our custom registration logic are both active.

### Step 4 — verify abstract enforcement

The metaclass inherited from `ABCMeta`, so abstract methods still behave normally.

In [46]:
try:
    Serializer()
except TypeError as exc:
    abstract_error = str(exc)

abstract_error

"Can't instantiate abstract class Serializer without an implementation for abstract method 'dumps'"

This illustrates an important rule:

> When a framework already depends on a specialized metaclass, extend that metaclass rather than replacing it with an unrelated one.

# 10. Problem 9 — custom keywords in a class definition

Python allows extra keywords in a class header:

```python
class User(Model, table="users", audited=True):
    ...
```

Those options can be consumed by metaclass hooks.

### Goal

We want a metaclass that understands:

- `table=...`
- `audited=True/False`

and stores normalized metadata on the created class.

### Step 1 — receive `**kwargs`

A robust metaclass often includes `**kwargs` in its hook signatures because class-creation options may be passed through the machinery.

In [47]:
class TableMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        allowed = {"table", "audited"}
        unknown = set(kwargs) - allowed

        if unknown:
            raise TypeError(
                f"Unknown class options: {sorted(unknown)}"
            )

        table = kwargs.pop(
            "table",
            name.lower(),
        )

        audited = bool(
            kwargs.pop("audited", False)
        )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.__table__ = table
        cls.__audited__ = audited

        return cls

### Step 2 — use custom class options

In [48]:
class UserRow(
    metaclass=TableMeta,
    table="users",
    audited=True,
):
    pass


class LogRow(metaclass=TableMeta):
    pass


(
    UserRow.__table__,
    UserRow.__audited__,
    LogRow.__table__,
    LogRow.__audited__,
)

('users', True, 'logrow', False)

The options are not ordinary class-body variables.

They are configuration inputs to class creation itself.

### Step 3 — reject typos

Rejecting unknown options is a useful best practice because misspelled configuration should fail loudly.

In [49]:
try:
    exec(
        '''
class BadRow(
    metaclass=TableMeta,
    tabel="oops",
):
    pass
''',
        globals(),
        {},
    )
except TypeError as exc:
    class_option_error = str(exc)

class_option_error

"Unknown class options: ['tabel']"

# 11. Problem 10 — build a small inheritance-aware record framework

This is the largest problem in the notebook.

We will build a tiny record system inspired by declarative frameworks.

A class will declare fields with annotations:

```python
class User(Record):
    id: int
    name: str
    active: bool = True
```

The metaclass will inspect those annotations and generate useful behavior.

### Requirements

The final framework should:

1. collect annotated fields,
2. inherit fields from base records,
3. preserve declaration order,
4. recognize class-level defaults,
5. generate `__init__`,
6. reject missing required fields,
7. reject unknown fields,
8. generate `__repr__`,
9. avoid overwriting user-defined `__init__` or `__repr__`,
10. expose field metadata on the class.

We will build this slowly instead of jumping directly to the final metaclass.

## 11.1 First step — inspect annotations

Python stores annotations in `__annotations__`.

In [50]:
class AnnotationExample:
    first: int
    second: str = "default"


AnnotationExample.__annotations__

{'first': int, 'second': str}

Notice that `first` is annotated but not assigned a class-level value.

`second` has both an annotation and a default value.

That gives us enough information to distinguish:

- required fields,
- optional fields with defaults.

## 11.2 Second step — define the framework base

The framework base itself has no fields.

Subclasses will inherit from it.

Before writing the complete metaclass, let's define the metadata we want each record class to expose:

```python
__record_fields__
__record_defaults__
```

`__record_fields__` will be a tuple preserving field order.

`__record_defaults__` will be a dictionary of fields that have defaults.

## 11.3 Third step — merge inherited metadata

Suppose:

```python
class Entity(Record):
    id: int

class User(Entity):
    name: str
```

`User` should contain both `id` and `name`.

Therefore metaclass `__new__` must inspect base classes before adding local annotations.

We will use ordinary dictionary insertion order.

Modern Python dictionaries preserve insertion order, which makes them suitable for this teaching example.

## 11.4 Fourth step — generate `__init__`

The generated initializer will accept keyword arguments.

Why keyword arguments?

Because they make validation straightforward and keep the implementation readable.

The algorithm is:

1. compute the allowed field names,
2. reject unknown keys,
3. determine which required fields are missing,
4. use user-provided values when available,
5. otherwise use defaults.

## 11.5 Fifth step — generate `__repr__`

The representation will display every declared field in order.

But only if the class author did not define a custom `__repr__`.

Now we have enough pieces to implement the complete metaclass.

In [51]:
class RecordMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)

        inherited_fields = []
        inherited_defaults = {}

        for base in bases:
            inherited_fields.extend(
                getattr(base, "__record_fields__", ())
            )
            inherited_defaults.update(
                getattr(base, "__record_defaults__", {})
            )

        local_annotations = dict(
            namespace.get("__annotations__", {})
        )

        # Start with inherited order.
        fields = list(inherited_fields)

        # Add local fields without duplicating inherited names.
        for field_name in local_annotations:
            if field_name not in fields:
                fields.append(field_name)

        defaults = dict(inherited_defaults)

        for field_name in local_annotations:
            if field_name in namespace:
                defaults[field_name] = namespace[field_name]

        fields = tuple(fields)

        namespace["__record_fields__"] = fields
        namespace["__record_defaults__"] = defaults

        if fields and "__init__" not in namespace:
            def __init__(self, **values):
                unknown = set(values) - set(fields)

                if unknown:
                    raise TypeError(
                        f"Unknown fields: {sorted(unknown)}"
                    )

                missing = [
                    field
                    for field in fields
                    if field not in values
                    and field not in defaults
                ]

                if missing:
                    raise TypeError(
                        f"Missing required fields: {missing}"
                    )

                for field in fields:
                    if field in values:
                        value = values[field]
                    else:
                        value = defaults[field]

                    setattr(self, field, value)

            namespace["__init__"] = __init__

        if fields and "__repr__" not in namespace:
            def __repr__(self):
                body = ", ".join(
                    f"{field}={getattr(self, field)!r}"
                    for field in fields
                )

                return (
                    f"{type(self).__name__}"
                    f"({body})"
                )

            namespace["__repr__"] = __repr__

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

## 11.6 Sixth step — create the framework base

In [52]:
class Record(metaclass=RecordMeta):
    pass

The base has no fields.

In [53]:
Record.__record_fields__, Record.__record_defaults__

((), {})

## 11.7 Seventh step — create an inherited record hierarchy

In [54]:
class EntityRecord(Record):
    id: int


class EmployeeRecord(EntityRecord):
    name: str
    department: str = "General"
    active: bool = True

Let's inspect the metadata before creating any instances.

In [55]:
(
    EntityRecord.__record_fields__,
    EmployeeRecord.__record_fields__,
    EmployeeRecord.__record_defaults__,
)

(('id',),
 ('id', 'name', 'department', 'active'),
 {'department': 'General', 'active': True})

The subclass inherited `id`, then appended its local fields in declaration order.

## 11.8 Eighth step — create a valid instance

In [56]:
employee = EmployeeRecord(
    id=101,
    name="Mira",
)

employee

EmployeeRecord(id=101, name='Mira', department='General', active=True)

The optional fields came from their defaults.

In [57]:
employee.department, employee.active

('General', True)

## 11.9 Ninth step — reject a missing required field

In [58]:
try:
    EmployeeRecord(name="No ID")
except TypeError as exc:
    missing_field_error = str(exc)

missing_field_error

"Missing required fields: ['id']"

## 11.10 Tenth step — reject an unknown field

In [59]:
try:
    EmployeeRecord(
        id=1,
        name="Test",
        nickname="unexpected",
    )
except TypeError as exc:
    unknown_field_error = str(exc)

unknown_field_error

"Unknown fields: ['nickname']"

## 11.11 Eleventh step — preserve a custom `__repr__`

A framework should usually avoid replacing behavior explicitly provided by the user.

In [60]:
class SecretRecord(Record):
    value: str

    def __repr__(self):
        return "<SecretRecord hidden>"


secret = SecretRecord(value="classified")

repr(secret)

'<SecretRecord hidden>'

The custom representation survived because the metaclass injected a default only when `__repr__` was absent from the namespace.

# 12. Problem 11 — a metaclass conflict and how to solve it

Multiple inheritance can introduce a new problem.

Suppose two base classes use unrelated metaclasses.

In [61]:
class MetaLeft(type):
    pass


class MetaRight(type):
    pass


class LeftBase(metaclass=MetaLeft):
    pass


class RightBase(metaclass=MetaRight):
    pass

What happens if we inherit from both?

In [62]:
try:
    class Conflict(LeftBase, RightBase):
        pass
except TypeError as exc:
    conflict_error = str(exc)

conflict_error

'metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases'

Python needs one metaclass compatible with the metaclasses of all base classes.

Neither `MetaLeft` nor `MetaRight` is a subclass of the other, so Python cannot choose automatically.

### Solution — create a combined metaclass

A combined metaclass can inherit from both.

In [63]:
class CombinedMeta(MetaLeft, MetaRight):
    pass


class Combined(
    LeftBase,
    RightBase,
    metaclass=CombinedMeta,
):
    pass


type(Combined)

__main__.CombinedMeta

Now the selected metaclass is a subclass of both required metaclasses.

In [64]:
(
    isinstance(Combined, MetaLeft),
    isinstance(Combined, MetaRight),
)

(True, True)

### Best-practice implication

Metaclasses are part of a class hierarchy's compatibility surface.

A framework that introduces a custom metaclass should consider whether users may need to combine it with other frameworks.

# 13. Problem 12 — should this even be a metaclass?

Advanced Python is not about using the most powerful mechanism.

It is about choosing the **simplest mechanism that correctly expresses the requirement**.

Let's compare three approaches to subclass registration.

## 13.1 Metaclass approach

In [65]:
class RegistrationMeta(type):
    subclasses = []

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        if bases:
            mcls.subclasses.append(cls)

        return cls


class MetaRegisteredBase(metaclass=RegistrationMeta):
    pass


class MetaRegisteredChild(MetaRegisteredBase):
    pass

This works.

But registration alone does not necessarily require a metaclass.

## 13.2 `__init_subclass__` approach

A base class can automatically receive notification whenever it is subclassed.

In [66]:
class InitSubclassBase:
    subclasses = []

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        InitSubclassBase.subclasses.append(cls)


class InitSubclassChild(InitSubclassBase):
    pass

For this requirement, `__init_subclass__` is arguably simpler and introduces fewer compatibility concerns.

## 13.3 Class decorator approach

A decorator is useful when registration should be explicit and opt-in.

In [67]:
decorated_registry = []


def register_class(cls):
    decorated_registry.append(cls)
    return cls


@register_class
class DecoratedCommand:
    pass

### Compare the results

In [68]:
(
    [cls.__name__ for cls in RegistrationMeta.subclasses],
    [cls.__name__ for cls in InitSubclassBase.subclasses],
    [cls.__name__ for cls in decorated_registry],
)

(['MetaRegisteredChild'], ['InitSubclassChild'], ['DecoratedCommand'])

### Decision guide

Use a **class decorator** when:

- the behavior is explicit and opt-in,
- post-processing the finished class is enough.

Use **`__init_subclass__`** when:

- the behavior naturally applies to subclasses of one base class,
- you do not need to control the class namespace before creation.

Use a **metaclass** when:

- you need `__prepare__`,
- you need to alter class construction centrally,
- you need metaclass `__call__`,
- you need class-definition keywords at the metaclass level,
- or the framework's semantics truly operate at the "class of classes" level.

# 14. Debugging workshop

The following short problems focus on mistakes that are easy to make when first inheriting from `type`.

## Debugging problem A — forgetting to return the class object

Consider:

```python
class BrokenMeta(type):
    def __new__(mcls, name, bases, namespace):
        super().__new__(mcls, name, bases, namespace)
```

What is wrong?

`__new__` must return the constructed object.

In [69]:
class CorrectReturnMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        return cls


class ReturnExample(metaclass=CorrectReturnMeta):
    pass


type(ReturnExample)

__main__.CorrectReturnMeta

## Debugging problem B — calling `type.__new__` with the wrong first argument

This is suspicious:

```python
type.__new__(type, name, bases, namespace)
```

because it explicitly asks `type` to create the class.

If we are writing a subclass of `type`, we usually want the active metaclass to remain involved.

The cooperative form is clearer:

```python
super().__new__(mcls, name, bases, namespace, **kwargs)
```

## Debugging problem C — late binding in generated functions

Suppose a loop creates several methods and every method refers to the loop variable.

If we are not careful, every generated method may end up using the final loop value.

In [70]:
functions = []

for value in range(3):
    def bad():
        return value

    functions.append(bad)

[f() for f in functions]

[2, 2, 2]

All functions return the last value.

A common fix is to freeze the current value using a default argument.

In [71]:
functions = []

for value in range(3):
    def good(value=value):
        return value

    functions.append(good)

[f() for f in functions]

[0, 1, 2]

This matters when metaclasses generate properties or methods inside loops.

# 15. Challenge set

The following problems are shorter.

Try to solve each one before reading the solution cell.

## Challenge 1 — automatic class IDs

Create a metaclass that assigns a monotonically increasing integer `class_id` to every class it creates.

The framework base should also receive an ID.

### Solution

In [72]:
class ClassIDMeta(type):
    next_id = 1

    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace["class_id"] = mcls.next_id
        mcls.next_id += 1

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )


class IDBase(metaclass=ClassIDMeta):
    pass


class IDOne(IDBase):
    pass


class IDTwo(IDBase):
    pass


(IDBase.class_id, IDOne.class_id, IDTwo.class_id)

(1, 2, 3)

## Challenge 2 — forbid public class attributes after creation

Allow attributes declared in the class body.

After the class has been created, prevent adding **new** public class attributes.

Existing public attributes may still be changed.

### Hint

Assignments such as:

```python
MyClass.new_name = 10
```

are handled by the metaclass's `__setattr__`.

### Solution

In [73]:
class LockedMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        type.__setattr__(cls, "__locked__", True)
        return cls

    def __setattr__(cls, name, value):
        locked = cls.__dict__.get("__locked__", False)
        is_public = not name.startswith("_")
        is_new = name not in cls.__dict__

        if locked and is_public and is_new:
            raise AttributeError(
                f"Cannot add new public attribute {name!r}"
            )

        super().__setattr__(name, value)


class LockedConfig(metaclass=LockedMeta):
    timeout = 10


LockedConfig.timeout = 20

try:
    LockedConfig.retries = 3
except AttributeError as exc:
    locked_error = str(exc)

LockedConfig.timeout, locked_error

(20, "Cannot add new public attribute 'retries'")

## Challenge 3 — instance counter

Create a metaclass that counts successful instance creations.

If instance initialization raises an exception, the counter must not increase.

### Solution

In [74]:
class InstanceCounterMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.instances_created = 0
        return cls

    def __call__(cls, *args, **kwargs):
        instance = super().__call__(*args, **kwargs)
        cls.instances_created += 1
        return instance


class Job(metaclass=InstanceCounterMeta):
    def __init__(self, valid=True):
        if not valid:
            raise ValueError("invalid job")


Job()
Job()

try:
    Job(valid=False)
except ValueError:
    pass

Job.instances_created

2

The count is `2`, not `3`, because it increments only after `super().__call__` successfully returns an instance.

# 16. Final self-check

The following cell performs a compact set of assertions across the notebook.

If it runs without raising an exception, the major examples are behaving as intended.

In [75]:
assert UserAccount.label == "User Account"
assert APIClient.label == "External API Client"

assert UserCreated.event_type == "user_created"

assert StrictSettings.timeout == 30

assert Symbol("USD") is Symbol("USD")
assert Symbol("USD") is not Symbol("EUR")

assert CommandMeta.get("save") is SaveCommand
assert CommandMeta.get("delete") is DeleteCommand

assert product.name == "Keyboard"
assert product.price == 99.0

assert Calculator.__method_call_counts__["add"] == 2
assert Calculator.__method_call_counts__["multiply"] == 1

assert SerializerMeta.registry["TextSerializer"] is TextSerializer
assert SerializerMeta.registry["ReprSerializer"] is ReprSerializer

assert UserRow.__table__ == "users"
assert UserRow.__audited__ is True

assert EmployeeRecord.__record_fields__ == (
    "id",
    "name",
    "department",
    "active",
)

assert employee.id == 101
assert employee.department == "General"
assert employee.active is True

assert isinstance(Combined, MetaLeft)
assert isinstance(Combined, MetaRight)

assert Job.instances_created == 2

"All self-checks passed."

'All self-checks passed.'

# 17. Final summary

By inheriting from `type`, we gained control over several distinct stages of Python's object model.

### `__prepare__`

Use it when you need to control the namespace **while the class body is executing**.

Example from this notebook:

- rejecting duplicate class attributes.

### metaclass `__new__`

Use it when you need to create, validate, transform, or enrich the class object.

Examples from this notebook:

- injecting labels,
- validating event definitions,
- registering command classes,
- generating descriptors,
- wrapping methods,
- building record metadata.

### metaclass `__call__`

Use it when you need to control what happens when the created class is called.

Examples from this notebook:

- flyweight instance reuse,
- counting successful instance construction.

### specialized metaclass inheritance

When another framework already uses a metaclass such as `ABCMeta`, extend that metaclass rather than introducing an unrelated one.

### class-definition keywords

Metaclasses can consume options written directly in the class header.

### metaclass conflicts

Multiple inheritance can require a combined metaclass compatible with all base-class metaclasses.

# 18. Best-practice checklist

Before introducing a custom metaclass, ask:

1. Could a class decorator solve this more clearly?
2. Could `__init_subclass__` solve it with less machinery?
3. Do I truly need access to class construction?
4. Do I need a custom namespace from `__prepare__`?
5. Do I need to customize instance creation through metaclass `__call__`?
6. Will this metaclass need to compose with `ABCMeta` or another framework metaclass?

When writing the metaclass:

- use cooperative `super()`,
- accept and forward `**kwargs`,
- avoid silently overwriting user-defined behavior,
- validate early with clear error messages,
- keep transformation helpers outside the metaclass where practical,
- avoid mutating base-class configuration accidentally,
- test invalid class definitions as well as valid ones,
- document hidden behavior because metaclass effects are not always obvious at the class definition site.

# 19. Review questions

Try answering these from memory.

1. Why is a normal class usually an instance of `type`?
2. What does the first parameter of metaclass `__new__` represent?
3. Why can `__prepare__` detect duplicates that `__new__` cannot?
4. Why is metaclass `__call__` a natural place for singleton or flyweight logic?
5. Why should registries usually reject duplicate keys?
6. Why should a framework avoid overwriting user-defined methods without an explicit contract?
7. Why can `ABCMeta` matter when designing a custom metaclass?
8. What causes a metaclass conflict?
9. Why is `super()` important in metaclass code?
10. When is `__init_subclass__` preferable to a metaclass?

# 20. Compact answer key

1. Because `type` is Python's default class-construction class.
2. It is the metaclass object currently creating the new class.
3. `__prepare__` controls the mapping during class-body execution, before duplicate assignments are lost.
4. Calling a class goes through its metaclass's `__call__`, so that hook controls instance construction.
5. Silent replacement makes framework behavior ambiguous and difficult to debug.
6. Explicit user behavior should normally take precedence over generated defaults.
7. `ABC` relies on `ABCMeta`; a custom metaclass may need to inherit from it for compatibility.
8. Python cannot find one metaclass that is compatible with the metaclasses required by all base classes.
9. Cooperative `super()` lets multiple compatible metaclasses participate according to the MRO.
10. When the desired behavior is naturally tied to subclasses of one base class and does not require deeper class-construction control.